# Get to Know a Dataset: StreetSpring Business Survivability Datasets

This notebook serves as a guided tour of the [StreetSpring Business Survivability Datasets](https://registry.opendata.aws/streetspring-survivability-datasets/). More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

The datasets give the projected chance, in percent, that a business of a given type lasts two or more years at a given place: neighborhoods and towns in 24 U.S. metropolitan areas, plus a national file that compares the metros. Every score in the files is on that scale. Each row describes a place in two ways:

- `avg_survivability_score`: the chance at a **typical** address in the place;
- `p90_survivability_score`: the chance at a **good** address, the 90th percentile of the scored locations inside the place. The rank columns are ordered on this one.

Reference material: the [full documentation and data dictionary](https://github.com/StreetSpringAI/open-data-examples/blob/main/streetspring-survivability/DOCUMENTATION.md), the [README that ships with the files](https://streetspring.com/resources/data/README.md) and the [methodology](https://streetspring.com/resources/tutorial/aeo-methodology). License: [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).

### Q: How have you organized your dataset? Help us understand the key prefix structure of your S3 bucket.

The dataset sits under a single prefix, `open-data/`, in the S3 bucket `streetspring-seo` (region `us-east-1`). There are no deeper prefixes. Under it you will find:

1. One **national file**, `national-survivability-scores-2026.csv`: one row per metro and business type.
2. 24 **metro files**, `<metro>-survivability-scores-2026.csv` (for example `philadelphia-survivability-scores-2026.csv`): one row per place, a neighborhood or a town, and business type.
3. `README.md` (column dictionary and worked recipes), `CHANGELOG.md` (every version and what changed), `LICENSE` (CC BY 4.0) and `CITATION.cff`.

Every CSV is self-describing. It opens with comment lines, each starting with `#`, that repeat the citation, the license, the version, the model version, the scale and worked recipes, so a single file still makes sense when it is shared on its own.

Full documentation for this dataset: https://github.com/StreetSpringAI/open-data-examples/blob/main/streetspring-survivability/DOCUMENTATION.md

> If the dataset moves to another bucket, change `BUCKET`, `PREFIX` and `REGION` in the cell below. Nothing else in the notebook needs to change.

First we will import the Python libraries required throughout this notebook.

In [ ]:
# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):
#
# boto3 >= 1.34
# pandas >= 2.0
# matplotlib >= 3.10

# Built-ins
import io
# Installed libraries
import boto3
import pandas as pd
import matplotlib.pyplot as plt
from botocore import UNSIGNED
from botocore.config import Config

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 60)

Next, we define the location of the dataset, create a boto3 S3 client and list everything under the prefix. The bucket is public, so the client sends unsigned requests: no AWS account or credentials are needed.

In [ ]:
# Location of the dataset
BUCKET = "streetspring-seo"
PREFIX = "open-data/"
REGION = "us-east-1"

# Unsigned requests: this is a public bucket
s3 = boto3.client("s3", region_name=REGION, config=Config(signature_version=UNSIGNED))

# Are there any deeper prefixes under open-data/?
top = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX, Delimiter="/")
print("Sub-prefixes under", PREFIX, ":", [p["Prefix"] for p in top.get("CommonPrefixes", [])] or "none")

# Every object under the prefix, with its size
objects = s3.list_objects_v2(Bucket=BUCKET, Prefix=PREFIX)["Contents"]
listing = pd.DataFrame(
    [{"key": o["Key"].removeprefix(PREFIX), "MB": round(o["Size"] / 1e6, 2), "last_modified": o["LastModified"].date()}
     for o in objects]
)
print(f"{len(listing)} objects, {listing.MB.sum():.0f} MB in total")
listing

The data file names follow one pattern, so the list of metros can be read straight from the keys.

In [ ]:
SUFFIX = "-survivability-scores-2026.csv"
csv_keys = sorted(k for k in listing.key if k.endswith(SUFFIX))
metros = [k.removesuffix(SUFFIX) for k in csv_keys if not k.startswith("national")]
print(f"{len(csv_keys)} CSV files: 1 national file and {len(metros)} metro files")
print(", ".join(metros))

The changelog travels with the files, and its first entry is the current version. Every file header carries the same version string, so you can always tell which release a file came from.

In [ ]:
def read_text(key):
    """Download one object under PREFIX as text."""
    return s3.get_object(Bucket=BUCKET, Key=PREFIX + key)["Body"].read().decode("utf-8")

changelog = read_text("CHANGELOG.md")
entries = changelog.split("\n## ")
print("Current version:", entries[1].splitlines()[0])
print("All versions:   ", ", ".join(e.splitlines()[0] for e in entries[1:]))

### Q: What data formats are present in your dataset? What kinds of data are stored using these formats? Can you give any advice for how you work with these data formats?

Every data file is a CSV (comma-separated values) in plain ASCII text with one header row, preceded by the `#` comment lines described above. The metro files share one set of columns and the national file has its own; both are documented column by column in the full documentation.

We chose CSV because many of the people who use these files (reporters, planners, prospective business owners, students) open them in a spreadsheet rather than in code, and each file is small enough to do that. The comment header keeps the license, the citation and the meaning of the score attached to a file when it is emailed or re-uploaded on its own.

Advice for working with the files:

- **Reading.** In pandas, `pd.read_csv(path_or_url, comment="#")` skips the header lines; in R, `read.csv(url, comment.char = "#")`. No data row contains a `#`, so the comment marker never cuts a value short. The helper below does the same split explicitly and also keeps the header lines.
- **Empty cells.** An empty cell means no value (not measured, not applicable, or not ranked). It never means zero.
- **Whole numbers.** In the metro files, whole-number columns that can be empty (ranks, counts) are written like `12.0`. Cast them to a nullable integer type if you need integers.
- **Units.** Score columns are percentages from 0 to 100. Census context columns such as `poverty_rate` and `vacancy_rate` are fractions of 1, and `vacancy_rate` is housing vacancy, not commercial vacancy.
- **Links.** URL columns hold paths on https://streetspring.com, except `methodology_url`, which is absolute.
- **AWS services.** The files can be read anonymously from Amazon S3 from anywhere, including notebooks in Amazon SageMaker Studio or Amazon SageMaker Studio Lab. If you query them in place with a SQL engine such as Amazon Athena, remember that an engine that does not understand comment lines will read them as rows. The number of comment lines differs between files, so filter out rows whose first column starts with `#` rather than skipping a fixed count.

### Q: Can you show us an example of downloading and loading data from your dataset?

We will write one small helper that downloads a file, separates the comment header from the table, and returns both. Then we load Philadelphia and print its header.

In [ ]:
def load(key):
    """Download one dataset file. Returns (header comment lines, DataFrame)."""
    lines = read_text(key).splitlines(keepends=True)
    n = 0
    while n < len(lines) and lines[n].startswith("#"):
        n += 1
    header = [line.rstrip("\n") for line in lines[:n]]
    df = pd.read_csv(io.StringIO("".join(lines[n:])), low_memory=False)
    return header, df

header, phl = load("philadelphia" + SUFFIX)
for line in header:
    print(line if len(line) <= 150 else line[:150] + " ...")

The header states the source, license, citation, model version, the ranking rules, the scale (`# Scale:`) and the version. Below it is the table: one row per business type and place. Transposing the first rows is a quick way to see all of the columns with example values.

In [ ]:
print(f"{len(phl):,} rows x {phl.shape[1]} columns")
print(f"{phl.business_subtype.nunique()} business types, {phl.neighborhood.nunique()} places")
print(phl.drop_duplicates("neighborhood").place_kind.value_counts().to_string())
phl.head(3).T

The columns fall into a few groups:

- **Identifiers:** `city`, `neighborhood` and `neighborhood_name` (the place), `place_kind` (`neighborhood` or `town`), `business_subtype` and `business_type_lookup` (the business type).
- **Scores:** `avg_survivability_score` (typical address), `p90_survivability_score` (good address), `max_` and `min_survivability_score` (best and weakest scored address), four confidence bounds, and the two failure-rate columns (100 minus the max and min scores).
- **Ranks and tiers:** the rank of the place for the business type, the rank of the business type in the place, their tiers, list lengths, and an overall rank for the place. They are empty for places or business types that are not ranked.
- **Context:** U.S. Census American Community Survey measures for the place (income, poverty, education, age, household size, commute, housing age, housing vacancy) and `competitor_count`.
- **Provenance:** links to the page on streetspring.com that shows each figure, the methodology link, and `scored_locations_in_neighborhood`, the number of scored locations behind the place's scores.

**Recipe 1: where in Philadelphia is an Italian restaurant most likely to last?** Filter to the business type and to neighborhoods (towns are ranked in their own list), then sort by the rank.

In [ ]:
cols = ["neighborhood_name", "p90_survivability_score", "avg_survivability_score",
        "neighborhood_rank_for_business_subtype", "tier_of_neighborhood_for_business_subtype",
        "total_ranked_neighborhoods_for_business_subtype", "scored_locations_in_neighborhood"]
italian = phl[(phl.business_subtype == "italian-restaurant") & (phl.place_kind == "neighborhood")]
italian.dropna(subset=["neighborhood_rank_for_business_subtype"]).sort_values(
    "neighborhood_rank_for_business_subtype")[cols].head(10)

**Recipe 2: what is most likely to last in one neighborhood?** Filter to the place and sort by the rank of the business type inside it. Here, Fishtown.

In [ ]:
fishtown = phl[phl.neighborhood == "fishtown"]
fishtown.sort_values("business_subtype_rank_for_neighborhood")[
    ["business_type_lookup", "p90_survivability_score", "avg_survivability_score",
     "business_subtype_rank_for_neighborhood", "tier_of_business_subtype_for_neighborhood"]].head(10)

**The national file compares metros.** Each row is one metro and business type. Which metro is best for a hot pot restaurant?

In [ ]:
_, national = load("national" + SUFFIX)
print(f"{len(national):,} rows x {national.shape[1]} columns, {national.city.nunique()} metros, "
      f"{national.business_subtype.nunique()} business types")
hot_pot = national[national.business_subtype == "hot-pot-restaurant"].sort_values("city_rank_for_business_subtype")
hot_pot[["city", "p90_survivability_score_for_business_subtype_in_city",
         "avg_survivability_score_for_business_subtype_in_city",
         "city_rank_for_business_subtype", "tier_of_city_for_business_subtype"]]

### Q: A picture is worth a thousand words. Show us a visual (or several!) from your dataset that either illustrates something informative about your dataset, or that you think might excite someone to dig in further.

To look across metros we load all 24 metro files into one table. This downloads every metro file once; the total size is printed by the listing above.

In [ ]:
frames = []
for metro in metros:
    _, df = load(metro + SUFFIX)
    frames.append(df)
metro_df = pd.concat(frames, ignore_index=True)

# Rows that carry a rank: ranked places and ranked business types
ranked = metro_df[metro_df.neighborhood_rank_for_business_subtype.notna()]
print(f"{len(metro_df):,} rows in the metro files; {len(ranked):,} ranked rows in {ranked.city.nunique()} metros")
print(f"{metro_df.drop_duplicates(['city', 'neighborhood']).shape[0]:,} places; "
      f"{ranked.business_subtype.nunique()} of {metro_df.business_subtype.nunique()} business types are ranked")

**How much does the neighborhood matter for one business?** Each box below shows the chance of lasting two or more years at a good address for a coffee shop, across one metro's ranked neighborhoods (towns are left out because they are ranked in a separate list). Metros are sorted by their median neighborhood. San Antonio does not appear because it ranks no places (its file header explains why).

In [ ]:
TYPE = "coffee-shop"
sub = ranked[(ranked.business_subtype == TYPE) & (ranked.place_kind == "neighborhood")]
order = sub.groupby("city").p90_survivability_score.median().sort_values().index
data = [sub.loc[sub.city == c, "p90_survivability_score"] for c in order]

fig, ax = plt.subplots(figsize=(12, 6.5), dpi=100)
ax.boxplot(data, orientation="horizontal", widths=0.6, showfliers=True,
           flierprops=dict(marker="o", markersize=3, alpha=0.5),
           medianprops=dict(color="#c0392b", linewidth=1.5))
ax.set_yticks(range(1, len(order) + 1))
ax.set_yticklabels([f"{c} ({len(d)})" for c, d in zip(order, data)])
ax.set_xlabel("Projected chance of lasting two or more years at a good address (%)")
ax.set_title(f"{TYPE}: spread across each metro's ranked neighborhoods (number of neighborhoods in brackets)", loc="left")
ax.grid(axis="x", linestyle="--", alpha=0.3)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

g = sub.groupby("city").p90_survivability_score
medians = g.median()
within_range = (g.max() - g.min()).median()
within_p10_p90 = (g.quantile(0.9) - g.quantile(0.1)).median()
print(f"Across the {len(medians)} metros, the median neighborhood runs from {medians.min():.1f} to {medians.max():.1f} "
      f"(a range of {medians.max() - medians.min():.1f} points; {medians.quantile(0.9) - medians.quantile(0.1):.1f} points "
      f"from the 10th to the 90th percentile metro).")
print(f"Inside the typical metro, neighborhoods span {within_range:.1f} points from best to worst and "
      f"{within_p10_p90:.1f} points from the 10th to the 90th percentile.")

### Q: What is one question that you have answered using these data? Can you show us how you came to that answer?

**Which matters most for a new business's projected chance of lasting two or more years: the type of business, the neighborhood, or the metro?**

We measure how far the good-address score moves when only one of the three changes:

1. **Business type:** within one ranked place, across the business types ranked there.
2. **Neighborhood:** within one metro and business type, across the ranked neighborhoods (towns excluded).
3. **Metro:** within one business type, across the metros. We measure this two ways, because a metro can be summarized at two levels: the metro-wide good-address score in the national file, and the median of the metro's ranked neighborhoods in the metro files.

The lists have different lengths, and a full range (best minus worst) grows with the length of the list. So we report both the full range and the spread between the 10th and 90th percentile, which does not grow with length.

In [ ]:
def spreads(groups, column):
    s = groups[column]
    return pd.DataFrame({
        "full range": s.max() - s.min(),
        "p10 to p90": s.quantile(0.9) - s.quantile(0.1),
        "items per list": s.size(),
    })

by_type = spreads(ranked.groupby(["city", "neighborhood"]), "p90_survivability_score")
by_place = spreads(ranked[ranked.place_kind == "neighborhood"].groupby(["city", "business_subtype"]),
                   "p90_survivability_score")
by_metro = spreads(national.groupby("business_subtype"), "p90_survivability_score_for_business_subtype_in_city")
# A second way to measure the metro: the median ranked neighborhood of each metro, compared across metros
metro_medians = (ranked[ranked.place_kind == "neighborhood"]
                 .groupby(["business_subtype", "city"]).p90_survivability_score.median().reset_index())
by_metro_nbhd = spreads(metro_medians.groupby("business_subtype"), "p90_survivability_score")

summary = pd.DataFrame({
    "what changes": ["business type (within a place)", "neighborhood (within a metro and type)",
                     "metro (within a type; national file)", "metro (within a type; median neighborhood)"],
    "lists measured": [len(d) for d in (by_type, by_place, by_metro, by_metro_nbhd)],
    "median items per list": [d["items per list"].median() for d in (by_type, by_place, by_metro, by_metro_nbhd)],
    "median full range (points)": [d["full range"].median() for d in (by_type, by_place, by_metro, by_metro_nbhd)],
    "median p10 to p90 (points)": [d["p10 to p90"].median() for d in (by_type, by_place, by_metro, by_metro_nbhd)],
}).round(1)
display(summary)

order_range = summary.sort_values("median full range (points)", ascending=False)["what changes"].tolist()
order_p10 = summary.sort_values("median p10 to p90 (points)", ascending=False)["what changes"].tolist()
print("Largest to smallest, full range:  ", " > ".join(order_range))
print("Largest to smallest, p10 to p90:  ", " > ".join(order_p10))

The same comparison as a picture: each box is the distribution of the p10 to p90 spread over all the lists of that kind.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5), dpi=100)
ax.boxplot([by_type["p10 to p90"], by_place["p10 to p90"], by_metro["p10 to p90"], by_metro_nbhd["p10 to p90"]],
           orientation="horizontal", widths=0.55,
           flierprops=dict(marker="o", markersize=2, alpha=0.3), medianprops=dict(color="#c0392b", linewidth=1.5))
ax.set_yticks([1, 2, 3, 4])
ax.set_yticklabels(["business type\n(within a place)", "neighborhood\n(within a metro and type)",
                    "metro, national file\n(within a type)", "metro, median neighborhood\n(within a type)"])
ax.invert_yaxis()   # first list on top
ax.set_xlabel("Spread of the good-address score, 10th to 90th percentile (points)")
ax.set_title("How far the projected chance moves when one thing changes", loc="left")
ax.grid(axis="x", linestyle="--", alpha=0.3)
for side in ("top", "right"):
    ax.spines[side].set_visible(False)
plt.tight_layout()
plt.show()

**What we found.** In this release the order is the same on both measures and with the metro measured either way: the business type moves the projected chance the most, the neighborhood next, and the metro least. It is a statement about the typical business type; for a single type, such as the coffee shop above, compare the numbers printed under its chart. For someone choosing where to open, the practical reading is that the choice of neighborhood inside a metro matters more than the choice between metros, and that the choice of business for a given neighborhood matters more than either.

**How far to trust it.**

- Part of the business-type spread comes from the way the scale is built: each type's scores are anchored to its sector's two-year survival benchmark from the U.S. Bureau of Labor Statistics (see the [methodology](https://streetspring.com/resources/tutorial/aeo-methodology)), and those benchmarks differ between sectors.
- The neighborhood spread is understated in Dallas, whose scores are pending a rescore after a defect in their Census inputs (see Known limits in the documentation).
- These are projections from one model version, not observed two-year outcomes.

### Q: What is one unanswered question that you think could be answered using these data? Do you have any recommendations or advice for someone wanting to answer this question?

**Challenge: for one business type, find the metro where the gap between its strong and weak neighborhoods is widest, and explain what local conditions separate them.**

The files show where the projected chance differs; they do not say why. The starter code below ranks every metro and business type pair by the p10 to p90 spread of its ranked neighborhoods, keeping pairs with at least 20 ranked neighborhoods so that a short list cannot produce a wide spread by chance. It then prints the strongest and weakest neighborhoods of the widest pair with the context columns the files carry.

Advice:

1. **Measure the spread fairly.** Use a percentile spread rather than best minus worst, and weight or filter places by `scored_locations_in_neighborhood`; a place with few scored locations has a noisier score.
2. **Beware circularity.** Neighborhood characteristics such as income, employment and housing are among the model's inputs, so a correlation between `poverty_rate` and the score is partly the model describing itself. Stronger explanations bring independent data: business licence or registration records, which several of the covered cities publish on their open data portals (for example Chicago and San Francisco); the Census Bureau's County Business Patterns and ZIP Code Business Patterns; transit service; zoning; commercial rents.
3. **Check the known limits** before building on a pair: Dallas scores are pending a rescore, San Antonio ranks nothing, and Pacific Palisades in Los Angeles is scored on inputs from before the January 2025 fire. If one business type dominates the list, read the changelog: four types (bar, home-improvement-store, irish-pub and middle-eastern-restaurant) had their scores restored in version 2026.09.25.1, and a spread that shows up mainly in recently restored types may say more about calibration than about places.
4. **Stretch goal: test it against what happened.** Where a city publishes dated business licence records, check whether the neighborhoods that score higher for a type kept more of that type's businesses open over two years. That would be independent evidence that the spread is real.

Share what you find by opening an issue or pull request on [StreetSpringAI/open-data-examples](https://github.com/StreetSpringAI/open-data-examples), or write to info@streetspring.com.

In [ ]:
MIN_PLACES = 20
nbhd = ranked[ranked.place_kind == "neighborhood"]
pairs = spreads(nbhd.groupby(["city", "business_subtype"]), "p90_survivability_score").reset_index()
pairs = pairs[pairs["items per list"] >= MIN_PLACES].sort_values("p10 to p90", ascending=False)
display(pairs.head(10).round(1))

city, btype = pairs.iloc[0][["city", "business_subtype"]]
context = ["neighborhood_name", "p90_survivability_score", "avg_survivability_score", "scored_locations_in_neighborhood",
           "income_metro_percentile", "poverty_rate", "pct_bachelor_plus", "vacancy_rate",
           "median_commute_minutes", "competitor_count"]
widest = nbhd[(nbhd.city == city) & (nbhd.business_subtype == btype)].sort_values("p90_survivability_score", ascending=False)
print(f"Widest pair: {btype} in {city}. Strongest and weakest ranked neighborhoods:")
display(pd.concat([widest.head(5), widest.tail(5)])[context])

### Citation and license

The data is licensed under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/). Please cite:

> StreetSpring (2026). StreetSpring Survivability Datasets 2026 (version as stated in the file header). https://streetspring.com/resources/datasets. DOI [10.5281/zenodo.22287996](https://doi.org/10.5281/zenodo.22287996). CC BY 4.0.

For a single figure, name the business type, the place and the year. Questions and corrections: info@streetspring.com.